# GSB 5544 — PA 3.2: Distances Between Observations — SOLUTION

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [1]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

The Ames data set (2,930 home sales in Ames, Iowa; tab-separated) is at the URL below. House 0 is the first row. The variables we need for part 1: `Gr Liv Area` (above-ground living area, sq ft), `Bedroom AbvGr`, `Full Bath`, `Half Bath`, and `SalePrice`; part 2 adds `House Style`.

In [2]:
df_ames = pd.read_csv("https://dlsun.github.io/pods/data/AmesHousing.txt", sep="\t")

df_ames.shape

(2930, 82)

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

**Plan.** (1) Build the three variables (combine full and half baths into one `Bathrooms` count). (2) **Standardize** each variable so a difference of one standard deviation counts the same for every variable — otherwise living area, measured in the thousands, swamps bedrooms and bathrooms, measured in ones. (3) Compute the distance from every house to house 0. (4) Keep only houses cheaper than house 0 and sort by distance.

In [3]:
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 * df_ames["Half Bath"]

house0 = df_ames.loc[0]
show = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "Year Built", "SalePrice"]
house0[show]

Gr Liv Area        1656
Bedroom AbvGr         3
Bathrooms           1.0
House Style      1Story
Neighborhood      NAmes
Year Built         1960
SalePrice        215000
Name: 0, dtype: object

In [4]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]

X = df_ames[features].astype(float)
X_z = (X - X.mean()) / X.std()          # standardized: every column has mean 0, SD 1

diff = X_z - X_z.loc[0]                 # row-wise difference from house 0
df_ames["dist_euclid"]    = np.sqrt((diff ** 2).sum(axis=1))
df_ames["dist_manhattan"] = diff.abs().sum(axis=1)

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
cheaper.sort_values("dist_euclid")[show + ["dist_euclid"]].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_euclid
1226,1661,3,1.0,SLvl,NAmes,1955,165500,0.009891
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.017804
1357,1666,3,1.0,2Story,OldTown,1925,161000,0.019782
758,1666,3,1.0,1.5Fin,IDOTRR,1927,135000,0.019782
291,1666,3,1.0,1.5Fin,SWISU,1931,100000,0.019782
2637,1668,3,1.0,1.5Fin,OldTown,1948,135000,0.023738
618,1644,3,1.0,1Story,NAmes,1953,167000,0.023738
2700,1640,3,1.0,1Story,Sawyer,1950,131000,0.031651
1529,1639,3,1.0,1.5Fin,SWISU,1936,115000,0.033629
179,1633,3,1.0,1.5Fin,OldTown,1948,129000,0.045499


In [5]:
cheaper.sort_values("dist_manhattan")[show + ["dist_manhattan"]].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_manhattan
1226,1661,3,1.0,SLvl,NAmes,1955,165500,0.009891
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.017804
291,1666,3,1.0,1.5Fin,SWISU,1931,100000,0.019782
758,1666,3,1.0,1.5Fin,IDOTRR,1927,135000,0.019782
1357,1666,3,1.0,2Story,OldTown,1925,161000,0.019782
2637,1668,3,1.0,1.5Fin,OldTown,1948,135000,0.023738
618,1644,3,1.0,1Story,NAmes,1953,167000,0.023738
2700,1640,3,1.0,1Story,Sawyer,1950,131000,0.031651
1529,1639,3,1.0,1.5Fin,SWISU,1936,115000,0.033629
179,1633,3,1.0,1.5Fin,OldTown,1948,129000,0.045499


**Sensitivity check.** Wrap the calculation in a function and compare the five nearest cheaper houses under different metrics and scalings.

In [6]:
def nearest_cheaper(features, i=0, metric="euclidean", scaling="z", k=5):
    """Indices of the k houses nearest to house i on `features`, among houses cheaper than house i."""
    X = df_ames[features].astype(float)
    if scaling == "z":
        X = (X - X.mean()) / X.std()
    elif scaling == "minmax":
        X = (X - X.min()) / (X.max() - X.min())
    diff = X - X.loc[i]
    dist = np.sqrt((diff ** 2).sum(axis=1)) if metric == "euclidean" else diff.abs().sum(axis=1)
    dist = dist[df_ames["SalePrice"] < df_ames.loc[i, "SalePrice"]]
    return dist.sort_values().head(k).index.tolist()

for scaling in ["z", "minmax", "none"]:
    for metric in ["euclidean", "manhattan"]:
        print(f"{scaling:>7} {metric:>10}: {nearest_cheaper(features, scaling=scaling, metric=metric)}")

      z  euclidean: [1226, 1940, 1357, 758, 291]
      z  manhattan: [1226, 1940, 291, 758, 1357]
 minmax  euclidean: [1226, 1940, 758, 291, 1357]
 minmax  manhattan: [1226, 1940, 291, 758, 1357]
   none  euclidean: [1550, 2638, 1927, 1197, 1293]
   none  manhattan: [1550, 655, 1493, 1197, 1293]


In [7]:
# What the *unscaled* distance picks: matches on square feet only, ignoring bedrooms and baths
df_ames.loc[nearest_cheaper(features, scaling="none"), show]

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice
1550,1656,3,1.5,SLvl,IDOTRR,1967,126000
2638,1657,4,1.0,1.5Fin,OldTown,1920,111500
1927,1657,3,2.0,1Story,NAmes,1970,163500
1197,1656,4,2.0,1Story,NWAmes,1973,135000
1293,1656,2,2.0,1.5Fin,OldTown,1940,119164


**Answer:** House 0 is a 1,656 sq ft, 3-bedroom, 1-bath one-story ranch in North Ames, built 1960, sold for $215,000. With standardized variables the nearest cheaper houses (rows 1226, 1940, 1357, 758, 291, …) are all 1,640–1,670 sq ft, 3 bedrooms, 1 bath, several of them also in North Ames and built in the 1950s–60s, selling for $100,000–$165,000 — sensible "same house, lower price" matches.

The results are **insensitive to the metric** (Euclidean and Manhattan return the same five houses in slightly different order) and to **z-score vs. min-max scaling**. They are **very sensitive to not scaling at all**: on raw units living area (SD ≈ 500 sq ft) dominates bedrooms and baths (SD < 1), so the unscaled distance just matches square footage and returns houses with different bedroom and bath counts.

**Sale price should not be in the distance.** The goal is a house that is *like* house 0 but *cheaper*: price is the constraint we filter on, not a dimension of similarity. Including it would pull the matches toward houses priced like house 0 — the opposite of what we want.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

`House Style` is categorical, so it cannot be subtracted. **One-hot encode** it (`pd.get_dummies`): one 0/1 column per style. Two houses with the same style differ by 0 on all those columns; two with different styles differ by 1 in two columns, which adds √2 ≈ 1.41 to the Euclidean distance — large compared with typical z-score differences, so a style mismatch is heavily penalized.

In [8]:
style_dummies = pd.get_dummies(df_ames["House Style"], dtype=float)
style_dummies.head()

,1.5Fin,1.5Unf,1Story,2.5Fin,2.5Unf,2Story,SFoyer,SLvl
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [9]:
X2 = pd.concat([X_z, style_dummies], axis=1)     # 3 standardized numeric columns + 8 style indicators

diff2 = X2 - X2.loc[0]
df_ames["dist_style"] = np.sqrt((diff2 ** 2).sum(axis=1))

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
cheaper.sort_values("dist_style")[show + ["dist_style"]].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,dist_style
1940,1647,3,1.0,1Story,NAmes,1953,153000,0.017804
618,1644,3,1.0,1Story,NAmes,1953,167000,0.023738
2700,1640,3,1.0,1Story,Sawyer,1950,131000,0.031651
314,1687,3,1.0,1Story,Timber,1948,160000,0.061324
788,1689,3,1.0,1Story,Edwards,1956,127500,0.065281
2282,1622,3,1.0,1Story,Mitchel,1961,168000,0.067259
2298,1608,3,1.0,1Story,Mitchel,1961,80000,0.094954
1240,1570,3,1.0,1Story,NAmes,1958,166800,0.170126
970,1771,3,1.0,1Story,Mitchel,1960,115000,0.227494
1410,1509,3,1.0,1Story,Edwards,1956,159900,0.290796


In [10]:
# Which styles did the part-1 top 10 have, and which do the part-2 top 10 have?
top1 = cheaper.sort_values("dist_euclid").head(10)["House Style"].value_counts()
top2 = cheaper.sort_values("dist_style").head(10)["House Style"].value_counts()
pd.DataFrame({"part 1 (no style)": top1, "part 2 (with style)": top2}).fillna(0).astype(int)

,part 1 (no style),part 2 (with style)
House Style,,
1.5Fin,5,0
1Story,3,10
2Story,1,0
SLvl,1,0


**Answer:** Every one of the ten nearest houses is now a `1Story` like house 0 (rows 1940, 618, 2700, 314, 788, …), whereas in part 1 several of the closest matches were 1.5- or 2-story houses that merely had similar square footage and room counts. Because a style mismatch costs √2 in distance, the dummies act almost like a filter: the algorithm first finds houses of the same style, then orders them by size and rooms. If that is too strict, scale the dummy columns down (multiply them by 0.5, say) so style is a preference rather than a requirement. Metric and scaling choices again barely change the list; the encoding of the categorical variable is what matters here.

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


**Variables chosen.** Quantitative: `Gr Liv Area`, `Bedroom AbvGr`, `Bathrooms`, `Year Built`, `Overall Qual`, `Lot Area`, `Garage Cars`. Categorical: `House Style`, `Neighborhood`, `Bldg Type`. These capture size, age, quality, land, and location — what a buyer who "likes house 0" is probably reacting to — without the dozens of near-duplicate basement/porch/garage columns. `Garage Cars` has one missing value; fill it with 0 so every house has a distance.

In [11]:
quant = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Year Built", "Overall Qual", "Lot Area", "Garage Cars"]
categ = ["House Style", "Neighborhood", "Bldg Type"]

X3 = pd.get_dummies(df_ames[quant + categ], columns=categ, dtype=float)
X3[quant] = (X3[quant] - X3[quant].mean()) / X3[quant].std()
X3 = X3.fillna(0)

diff3 = X3 - X3.loc[0]
df_ames["dist_full"] = np.sqrt((diff3 ** 2).sum(axis=1))

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
show3 = show + ["Overall Qual", "Lot Area", "Garage Cars", "Bldg Type"]
cheaper.sort_values("dist_full")[show3 + ["dist_full"]].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,Overall Qual,Lot Area,Garage Cars,Bldg Type,dist_full
1895,1652,3,1.5,1Story,NAmes,1959,200000,6,22002,2.0,1Fam,1.463780
1013,1474,3,1.0,1Story,Gilbert,1952,115000,6,31220,2.0,1Fam,1.484742
2223,1560,3,1.5,1Story,Crawfor,1960,201000,6,25485,2.0,1Fam,1.810282
2294,1676,3,1.5,1Story,Mitchel,1977,196000,5,33983,2.0,1Fam,1.871772
970,1771,3,1.0,1Story,Mitchel,1960,115000,5,21750,2.0,1Fam,2.042280
2700,1640,3,1.0,1Story,Sawyer,1950,131000,5,21370,2.0,1Fam,2.086728
639,1382,3,1.0,SLvl,NAmes,1962,176000,6,19296,2.0,1Fam,2.191809
1896,1429,3,1.0,1Story,NAmes,1960,181900,6,14585,2.0,1Fam,2.226585
2273,1572,3,1.5,1Story,Timber,1963,186700,5,35133,3.0,1Fam,2.248441
2590,2039,3,1.5,1Story,NAmes,1941,167000,7,21299,3.0,1Fam,2.360145


In [12]:
house0[show3]

Gr Liv Area        1656
Bedroom AbvGr         3
Bathrooms           1.0
House Style      1Story
Neighborhood      NAmes
Year Built         1960
SalePrice        215000
Overall Qual          6
Lot Area          31770
Garage Cars         2.0
Bldg Type          1Fam
Name: 0, dtype: object

In [13]:
# Sensitivity: drop Lot Area (house 0 sits on an unusually large lot) and see how much the list changes
quant_no_lot = [q for q in quant if q != "Lot Area"]
X4 = pd.get_dummies(df_ames[quant_no_lot + categ], columns=categ, dtype=float)
X4[quant_no_lot] = (X4[quant_no_lot] - X4[quant_no_lot].mean()) / X4[quant_no_lot].std()
X4 = X4.fillna(0)
df_ames["dist_no_lot"] = np.sqrt(((X4 - X4.loc[0]) ** 2).sum(axis=1))

cheaper = df_ames[df_ames["SalePrice"] < house0["SalePrice"]]
a = set(cheaper.sort_values("dist_full").head(10).index)
b = set(cheaper.sort_values("dist_no_lot").head(10).index)
print("top-10 overlap with / without Lot Area:", len(a & b), "of 10")
cheaper.sort_values("dist_no_lot")[show3 + ["dist_no_lot"]].head(10)

top-10 overlap with / without Lot Area: 2 of 10


,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,Overall Qual,Lot Area,Garage Cars,Bldg Type,dist_no_lot
1240,1570,3,1.0,1Story,NAmes,1958,166800,6,13200,2.0,1Fam,0.182525
618,1644,3,1.0,1Story,NAmes,1953,167000,6,9600,2.0,1Fam,0.232655
1896,1429,3,1.0,1Story,NAmes,1960,181900,6,14585,2.0,1Fam,0.449052
989,1414,3,1.0,1Story,NAmes,1958,176500,6,11029,2.0,1Fam,0.483271
1895,1652,3,1.5,1Story,NAmes,1959,200000,6,22002,2.0,1Fam,0.778503
1239,1261,3,1.0,1Story,NAmes,1958,163000,6,9120,2.0,1Fam,0.784184
147,1580,3,1.5,1Story,NAmes,1959,159500,6,10032,2.0,1Fam,0.792848
1216,1252,3,1.0,1Story,NAmes,1959,142000,6,10721,2.0,1Fam,0.799878
1231,1537,3,1.5,1Story,NAmes,1962,174000,6,8400,2.0,1Fam,0.815291
2558,1433,3,1.0,1Story,NAmes,1961,161000,5,9600,2.0,1Fam,0.835439


**Answer:** House 0 is a 1960 one-story, 3-bed, 1-bath, quality-5 house on a very large lot (31,770 sq ft — the 99th percentile). With size, age, quality, lot, garage, style, neighborhood, and building type all in the distance, the nearest cheaper houses are one-story single-family homes of 1,470–1,770 sq ft, 3 bedrooms, 1–1.5 baths, quality 5–6, on similarly huge lots (20,000–35,000 sq ft), mostly built in the 1950s–70s. They read as the same *kind* of property, which is what the added variables buy us.

Sensitivity is now noticeably higher than in parts 1–2: dropping `Lot Area` alone replaces most of the top ten, because the lot is house 0's most unusual feature and standardization makes "unusual" expensive to match. The general lesson: with many variables, *which* variables you include (and how you scale them) matters more than Euclidean vs. Manhattan. Sale price is still left out for the reason given in part 1.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [14]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [15]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

Both variables are quantitative but on wildly different scales (admission rate 0–1; undergraduates 5–119,000), so standardize before computing Euclidean distance to Cal Poly. The unscaled version is shown for contrast.

In [16]:
num = ["AdmissionRate", "Undergraduates"]

A = df_college[num]
A_z = (A - A.mean()) / A.std()
dist1 = np.sqrt(((A_z - A_z.loc[school_name]) ** 2).sum(axis=1))

df_college.assign(dist=dist1.round(3)).sort_values("dist")[num + ["CarnegieClassification", "Ownership", "dist"]].head(11)

,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,dist
Institution,,,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,Master's Colleges & Universities: Larger Programs,Public,0.000
University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,0.309
DeVry University-Illinois,0.4552,19729.0,Master's Colleges & Universities: Larger Programs,Private for-profit,0.593
University of North Carolina at Chapel Hill,0.2040,19722.0,Doctoral Universities: Very High Research Acti...,Public,0.597
Clemson University,0.4922,21577.0,Doctoral Universities: Very High Research Acti...,Public,0.737
University of Virginia-Main Campus,0.2074,17041.0,Doctoral Universities: Very High Research Acti...,Public,0.761
CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761
Stony Brook University,0.4806,17900.0,Doctoral Universities: Very High Research Acti...,Public,0.796
Boston University,0.1865,17501.0,Doctoral Universities: Very High Research Acti...,Private nonprofit,0.797


In [17]:
# Without scaling, Undergraduates (SD ≈ 7,800) dominates and AdmissionRate is effectively ignored
dist1_raw = np.sqrt(((A - A.loc[school_name]) ** 2).sum(axis=1))
df_college.assign(dist=dist1_raw.round(1)).sort_values("dist")[num + ["dist"]].head(6)

,AdmissionRate,Undergraduates,dist
Institution,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,0.0
University of Iowa,0.8621,21198.0,108.0
East Carolina University,0.9389,21231.0,141.0
Virginia Commonwealth University,0.9277,20918.0,172.0
University at Buffalo,0.7009,21303.0,213.0
University of Kentucky,0.9401,21358.0,268.0


**Answer:** Using **standardized Euclidean distance** on admission rate and undergraduate enrollment, the schools nearest Cal Poly (33% admit, 21,090 undergraduates) are UC Santa Barbara, DeVry University–Illinois, UNC Chapel Hill, Clemson, the University of Virginia, CUNY Hunter, Stony Brook, and Boston University — all selective-ish schools with roughly 15,000–23,000 undergraduates. Without scaling, the list becomes Iowa, East Carolina, VCU, Buffalo, and Kentucky: schools with ~21,000 undergraduates regardless of admission rate (many admit 70–80%), because a 0.4 difference in admit rate is invisible next to a difference of hundreds of students. Standardizing is the decision that makes both variables count.

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

Add one-hot columns for the two categorical variables. Note: `Institution` is not unique in this file (15 duplicated names), so instead of `pd.concat` on the index — which fails on duplicate labels — call `pd.get_dummies` on a single frame and pass `columns=` to encode only the categorical ones. Then standardize the two numeric columns as before.

In [18]:
B = pd.get_dummies(df_college[num + ["CarnegieClassification", "Ownership"]],
                   columns=["CarnegieClassification", "Ownership"], dtype=float)
B[num] = (B[num] - B[num].mean()) / B[num].std()

dist2 = np.sqrt(((B - B.loc[school_name]) ** 2).sum(axis=1))
df_college.assign(dist=dist2.round(3)).sort_values("dist")[num + ["CarnegieClassification", "Ownership", "dist"]].head(11)

,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,dist
Institution,,,,,
California Polytechnic State University-San Luis Obispo,0.3300,21090.0,Master's Colleges & Universities: Larger Programs,Public,0.000
CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761
CUNY Bernard M Baruch College,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.074
CUNY John Jay College of Criminal Justice,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.185
CUNY Brooklyn College,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376
University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.448
California State Polytechnic University-Pomona,0.6062,26802.0,Master's Colleges & Universities: Larger Programs,Public,1.450
CUNY Queens College,0.6078,14859.0,Master's Colleges & Universities: Larger Programs,Public,1.491
DeVry University-Illinois,0.4552,19729.0,Master's Colleges & Universities: Larger Programs,Private for-profit,1.534


**Answer:** Cal Poly is a *public* "Master's Colleges & Universities: Larger Programs" school, and the dummies make those two facts count for a lot (a mismatch on either adds √2 to the distance). The nearest schools are now CUNY Hunter, Baruch, John Jay, and Brooklyn College, then Cal Poly Pomona and CUNY Queens — public master's-level institutions with 12,000–27,000 undergraduates and moderate admission rates. UCSB, UNC, and UVA drop down the list because they are research doctoral universities, and DeVry drops because it is private for-profit. Whether that is "more similar" depends on the question: for *how selective and how big*, part 1's list is better; for *what kind of institution*, this one is.

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

The 38 `PCIP` columns are all proportions on the same 0–1 scale, so this time we do **not** standardize: doing so would inflate tiny, rare fields (a 1% difference in library science would count as much as a 20% difference in engineering). Plain Euclidean distance on the raw proportions — and, as a check, cosine similarity, which compares the *mix* regardless of magnitude — give the same neighbours.

In [19]:
P = df_college.filter(like="PCIP")
cp_fields = P.loc[school_name]
cp_fields.sort_values(ascending=False).head(6)      # 14 = engineering, 52 = business, 01 = agriculture, 45 = social sciences

PCIP14    0.2314
PCIP52    0.1637
PCIP01    0.1084
PCIP45    0.0588
PCIP26    0.0495
PCIP04    0.0441
Name: California Polytechnic State University-San Luis Obispo, dtype: float64

In [20]:
dist3 = np.sqrt(((P - cp_fields) ** 2).sum(axis=1))
df_college.assign(dist=dist3.round(3)).sort_values("dist")[["State", "Undergraduates", "CarnegieClassification", "dist"]].head(11)

,State,Undergraduates,CarnegieClassification,dist
Institution,,,,
California Polytechnic State University-San Luis Obispo,CA,21090.0,Master's Colleges & Universities: Larger Programs,0.000
North Carolina State University at Raleigh,NC,24999.0,Doctoral Universities: Very High Research Acti...,0.084
Iowa State University,IA,25537.0,Doctoral Universities: Very High Research Acti...,0.085
University of Illinois Urbana-Champaign,IL,33889.0,Doctoral Universities: Very High Research Acti...,0.111
Mississippi State University,MS,18451.0,Doctoral Universities: Very High Research Acti...,0.119
Texas A & M University-College Station,TX,56006.0,Doctoral Universities: Very High Research Acti...,0.124
Clemson University,SC,21577.0,Doctoral Universities: Very High Research Acti...,0.128
Purdue University-Main Campus,IN,37658.0,Doctoral Universities: Very High Research Acti...,0.134
Virginia Polytechnic Institute and State University,VA,29699.0,Doctoral Universities: Very High Research Acti...,0.138


In [21]:
# Cosine similarity as a cross-check (1 = identical mix of fields)
norms = np.sqrt((P ** 2).sum(axis=1))
cosine = (P @ cp_fields) / (norms * norms.loc[school_name])
cosine.sort_values(ascending=False).head(8).round(3)

Institution
California Polytechnic State University-San Luis Obispo    1.000
North Carolina State University at Raleigh                 0.969
Iowa State University                                      0.967
University of Illinois Urbana-Champaign                    0.941
Mississippi State University                               0.933
Texas A & M University-College Station                     0.927
Clemson University                                         0.923
Virginia Polytechnic Institute and State University        0.920
dtype: float64

**Answer:** Judged only by *what students study*, Cal Poly's neighbours are the big public land-grant universities: NC State, Iowa State, Illinois Urbana-Champaign, Mississippi State, Texas A&M, Clemson, Purdue, Virginia Tech, Auburn, and West Virginia. That is exactly Cal Poly's field mix — engineering (23%), business (16%), and agriculture (11%) — which is rare outside the land-grant system. None of these schools appeared in parts 1–2, because they are larger, doctoral, and mostly less selective; which list is "right" depends entirely on which notion of similarity you meant, and that choice — the variables, not the metric — is the real decision.